# SEED-IV RD target-domain monitoring (diagnostic only)

This notebook trains the current fold-specific RD representation with the masked Transformer and evaluates the held-out target subject every `TEST_EVERY` epochs. It reports target accuracy as **mean ± population standard deviation across target subjects**.

> **Warning — target peeking:** changing hyperparameters after looking at these target curves leaks test-domain information. Use this notebook to diagnose the large gap from the previous ~90% result, not to produce the final paper number. The official `scripts/train_rd.py` path remains source-validation-only.


In [1]:
from __future__ import annotations

import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'cmrd').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook inside the CMRD repository')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from cmrd.config import load_config
from cmrd.models import PlainTransformer
from cmrd.processed import load_split
from cmrd.training.engine import SequenceDataset, collate_sequences, evaluate, fit_normalizer
from cmrd.training.runtime import seed_everything

print('Project:', ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Project: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting
PyTorch: 2.13.0.dev20260422+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## Parameters
Edit this cell between diagnostic runs. Change `RUN_TAG` whenever parameters change so results are not mixed. Start with a few subjects such as `[1, 2, 3]`; use all 15 only after the code and settings look right.

In [2]:
CONFIG_PATH = ROOT / 'configs' / 'seediv' / 'rd.yaml'
TARGET_SUBJECTS = list(range(1, 16))  # e.g. [1, 2, 3] for a quick check

# Model
D_MODEL = 640
NHEAD = 8
LAYERS = 6
FEEDFORWARD = 512
DROPOUT = 0.2

# Optimization
EPOCHS = 100
TEST_EVERY = 10
BATCH_SIZE = 128
LEARNING_RATE = 1e-4
MINIMUM_LEARNING_RATE = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.0
GRADIENT_CLIP_NORM = 1.0
SEED = 42
DETERMINISTIC = True
DEVICE = 'cuda'  # explicit for the RTX 5080 Laptop
NUM_WORKERS = 0  # safest setting for Windows + Jupyter

# Output / resume
RUN_TAG = 'rd_d128_l2_lr3e-4_seed42'
RESUME = True
SAVE_CHECKPOINTS = True

assert EPOCHS > 0 and TEST_EVERY > 0
assert D_MODEL % NHEAD == 0
assert len(set(TARGET_SUBJECTS)) == len(TARGET_SUBJECTS)
assert all(1 <= subject <= 15 for subject in TARGET_SUBJECTS)
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable')

RUN_DIR = ROOT / 'runs' / 'diagnostics' / 'seediv_rd_target_monitor' / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Diagnostic output:', RUN_DIR)
print('TARGET PEEKING ENABLED: results from this notebook are not paper-final metrics.')


Diagnostic output: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting\runs\diagnostics\seediv_rd_target_monitor\rd_d128_l2_lr3e-4_seed42
TARGET PEEKING ENABLED: results from this notebook are not paper-final metrics.


In [3]:
CONFIG = load_config(CONFIG_PATH, expected_feature='rd')
DEVICE_OBJ = torch.device(DEVICE)

SETTINGS = {
    'config': str(CONFIG_PATH),
    'config_hash': CONFIG.hash(),
    'preprocessing_signature': CONFIG.preprocessing_signature(),
    'target_subjects': TARGET_SUBJECTS,
    'model': {'d_model': D_MODEL, 'nhead': NHEAD, 'layers': LAYERS, 'feedforward': FEEDFORWARD, 'dropout': DROPOUT},
    'training': {
        'epochs': EPOCHS, 'test_every': TEST_EVERY, 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'minimum_learning_rate': MINIMUM_LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY, 'label_smoothing': LABEL_SMOOTHING,
        'gradient_clip_norm': GRADIENT_CLIP_NORM, 'seed': SEED,
        'deterministic': DETERMINISTIC, 'device': DEVICE,
    },
    'diagnostic_target_peeking': True,
}
settings_path = RUN_DIR / 'settings.json'
if settings_path.exists():
    previous = json.loads(settings_path.read_text(encoding='utf-8'))
    if previous != SETTINGS:
        raise ValueError('RUN_TAG already contains different settings. Change RUN_TAG or restore the old parameters.')
else:
    settings_path.write_text(json.dumps(SETTINGS, indent=2, ensure_ascii=False), encoding='utf-8')

# Load one fold without target data first to inspect the current protocol.
sample_train, sample_validation, sample_target, sample_split = load_split(CONFIG, TARGET_SUBJECTS[0], include_test=False)
print('Example split:', sample_split)
print('Train/validation/target loaded:', len(sample_train), len(sample_validation), len(sample_target))
print('RD trial shape:', sample_train[0].x.shape, 'feature_dim=', sample_train[0].x.shape[1])
print('Labels:', sorted({sample.label for sample in sample_train + sample_validation}))
del sample_train, sample_validation, sample_target
gc.collect()


Example split: SubjectSplit(train_subjects=(3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15), validation_subjects=(2, 10), target_subject=1)
Train/validation/target loaded: 864 144 0
RD trial shape: (168, 310) feature_dim= 310
Labels: [0, 1, 2, 3]


1250

## Training helpers
The source-training normalization and source-validation split are unchanged. The only deliberate protocol violation is evaluating the held-out target subject every `TEST_EVERY` epochs.

In [4]:
def make_loader(samples, mean, std, shuffle: bool, seed: int) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        SequenceDataset(samples, mean, std),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE_OBJ.type == 'cuda',
        collate_fn=collate_sequences,
        generator=generator,
    )

def train_one_fold(target_subject: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    fold_epoch_path = RUN_DIR / f'fold-{target_subject:02d}_epochs.csv'
    fold_target_path = RUN_DIR / f'fold-{target_subject:02d}_target_curve.csv'
    if RESUME and fold_epoch_path.is_file() and fold_target_path.is_file():
        old_epochs = pd.read_csv(fold_epoch_path)
        old_target = pd.read_csv(fold_target_path)
        if len(old_epochs) == EPOCHS and int(old_epochs['epoch'].max()) == EPOCHS:
            print(f'fold={target_subject:02d} reused from disk')
            return old_epochs, old_target

    seed_everything(SEED, DETERMINISTIC)
    # Candidate settings are already fixed before target samples are loaded.
    train_samples, validation_samples, target_samples, split = load_split(CONFIG, target_subject, include_test=True)
    mean, std = fit_normalizer(train_samples)
    max_length = max(sample.x.shape[0] for sample in train_samples + validation_samples + target_samples)
    input_dim = train_samples[0].x.shape[1]

    train_loader = make_loader(train_samples, mean, std, True, SEED)
    validation_loader = make_loader(validation_samples, mean, std, False, SEED)
    target_loader = make_loader(target_samples, mean, std, False, SEED)
    model = PlainTransformer(
        input_dim=input_dim, classes=int(CONFIG.raw['dataset']['classes']), max_length=max_length,
        d_model=D_MODEL, nhead=NHEAD, layers=LAYERS, feedforward=FEEDFORWARD, dropout=DROPOUT,
    ).to(DEVICE_OBJ)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MINIMUM_LEARNING_RATE)

    epoch_rows = []
    target_rows = []
    best_source_score = (-float('inf'), -float('inf'))
    best_source_state = None
    best_source_epoch = 0
    started = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        seen = 0
        for data, mask, labels in train_loader:
            data = data.to(DEVICE_OBJ, non_blocking=True)
            mask = mask.to(DEVICE_OBJ, non_blocking=True)
            labels = labels.to(DEVICE_OBJ, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(data, mask), labels)
            loss.backward()
            if GRADIENT_CLIP_NORM > 0:
                nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            optimizer.step()
            loss_sum += float(loss.item()) * labels.shape[0]
            seen += labels.shape[0]
        scheduler.step()

        source_metrics = evaluate(model, validation_loader, DEVICE_OBJ, int(CONFIG.raw['dataset']['classes']))
        source_score = (float(source_metrics['macro_f1']), float(source_metrics['accuracy']))
        if source_score > best_source_score:
            best_source_score = source_score
            best_source_epoch = epoch
            best_source_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        row = {
            'target_subject': target_subject, 'epoch': epoch,
            'train_loss': loss_sum / max(seen, 1),
            'learning_rate': optimizer.param_groups[0]['lr'],
            'source_validation_accuracy': source_metrics['accuracy'],
            'source_validation_macro_f1': source_metrics['macro_f1'],
            'elapsed_seconds': time.perf_counter() - started,
        }
        epoch_rows.append(row)

        if epoch % TEST_EVERY == 0 or epoch == EPOCHS:
            target_metrics = evaluate(model, target_loader, DEVICE_OBJ, int(CONFIG.raw['dataset']['classes']))
            target_row = {
                **row,
                'target_accuracy': target_metrics['accuracy'],
                'target_balanced_accuracy': target_metrics['balanced_accuracy'],
                'target_macro_f1': target_metrics['macro_f1'],
                'target_confusion_matrix': json.dumps(target_metrics['confusion_matrix']),
            }
            target_rows.append(target_row)
            print(
                f'fold={target_subject:02d} epoch={epoch:03d} '
                f'loss={row["train_loss"]:.4f} source_val_f1={source_metrics["macro_f1"]:.4f} '
                f'target_acc={target_metrics["accuracy"]:.4f} target_f1={target_metrics["macro_f1"]:.4f}'
            )

    epoch_frame = pd.DataFrame(epoch_rows)
    target_frame = pd.DataFrame(target_rows)
    epoch_frame.to_csv(fold_epoch_path, index=False)
    target_frame.to_csv(fold_target_path, index=False)

    if SAVE_CHECKPOINTS and best_source_state is not None:
        torch.save(
            {
                'model_state_dict': best_source_state,
                'normalization_mean': mean, 'normalization_std': std,
                'best_source_validation_epoch': best_source_epoch,
                'best_source_validation_score': best_source_score,
                'target_subject': target_subject, 'split': split.as_dict(), 'settings': SETTINGS,
                'warning': 'Target was monitored during training; do not report as paper-final.',
            },
            RUN_DIR / f'fold-{target_subject:02d}_best-source-validation.pt',
        )

    del model, optimizer, scheduler, train_loader, validation_loader, target_loader
    del train_samples, validation_samples, target_samples
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return epoch_frame, target_frame


## Run selected LOSO folds
This trains one independent model per target subject. After each fold, the running table shows mean ± std across the target subjects completed so far. The final table is meaningful only when `n_subjects == 15`.

In [5]:
all_epoch_frames = []
all_target_frames = []

for target_subject in TARGET_SUBJECTS:
    epoch_frame, target_frame = train_one_fold(target_subject)
    all_epoch_frames.append(epoch_frame)
    all_target_frames.append(target_frame)
    running = pd.concat(all_target_frames, ignore_index=True)
    running_summary = (
        running.groupby('epoch', as_index=False)
        .agg(
            target_acc_mean=('target_accuracy', 'mean'),
            target_acc_std=('target_accuracy', lambda values: values.std(ddof=0)),
            target_macro_f1_mean=('target_macro_f1', 'mean'),
            target_macro_f1_std=('target_macro_f1', lambda values: values.std(ddof=0)),
            n_subjects=('target_subject', 'nunique'),
        )
    )
    latest = running_summary.iloc[-1]
    print(
        f'RUNNING epoch={int(latest.epoch):03d}: target ACC '
        f'{latest.target_acc_mean:.4f} ± {latest.target_acc_std:.4f} '
        f'across {int(latest.n_subjects)} subject(s)'
    )


fold=01 epoch=010 loss=0.8612 source_val_f1=0.5334 target_acc=0.5556 target_f1=0.5229
fold=01 epoch=020 loss=0.1295 source_val_f1=0.5185 target_acc=0.6111 target_f1=0.6080
fold=01 epoch=030 loss=0.0274 source_val_f1=0.5139 target_acc=0.5556 target_f1=0.5490
fold=01 epoch=040 loss=0.0122 source_val_f1=0.5118 target_acc=0.6111 target_f1=0.6095
fold=01 epoch=050 loss=0.0001 source_val_f1=0.5315 target_acc=0.6111 target_f1=0.6083
fold=01 epoch=060 loss=0.0001 source_val_f1=0.5399 target_acc=0.6250 target_f1=0.6217


KeyboardInterrupt: 

## Aggregate every-10-epoch target curve

In [ ]:
epoch_results = pd.concat(all_epoch_frames, ignore_index=True)
target_results = pd.concat(all_target_frames, ignore_index=True)
target_summary = (
    target_results.groupby('epoch', as_index=False)
    .agg(
        target_acc_mean=('target_accuracy', 'mean'),
        target_acc_std=('target_accuracy', lambda values: values.std(ddof=0)),
        target_acc_min=('target_accuracy', 'min'),
        target_acc_max=('target_accuracy', 'max'),
        target_macro_f1_mean=('target_macro_f1', 'mean'),
        target_macro_f1_std=('target_macro_f1', lambda values: values.std(ddof=0)),
        source_validation_acc_mean=('source_validation_accuracy', 'mean'),
        source_validation_macro_f1_mean=('source_validation_macro_f1', 'mean'),
        n_subjects=('target_subject', 'nunique'),
    )
)

epoch_results.to_csv(RUN_DIR / 'all_fold_epochs.csv', index=False)
target_results.to_csv(RUN_DIR / 'all_fold_target_curve.csv', index=False)
target_summary.to_csv(RUN_DIR / 'target_mean_std_by_epoch.csv', index=False)

display(target_summary.style.format({
    'target_acc_mean': '{:.2%}', 'target_acc_std': '{:.2%}',
    'target_acc_min': '{:.2%}', 'target_acc_max': '{:.2%}',
    'target_macro_f1_mean': '{:.2%}', 'target_macro_f1_std': '{:.2%}',
    'source_validation_acc_mean': '{:.2%}', 'source_validation_macro_f1_mean': '{:.2%}',
}))

best_target_row = target_summary.loc[target_summary['target_acc_mean'].idxmax()]
print(
    f'Exploratory best target checkpoint: epoch={int(best_target_row.epoch)}, '
    f'ACC={best_target_row.target_acc_mean:.2%} ± {best_target_row.target_acc_std:.2%}; '
    'this epoch is target-selected and must not be reported as a clean test result.'
)


In [ ]:
try:
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(9, 5))
    axis.plot(target_summary['epoch'], target_summary['target_acc_mean'], marker='o', label='Target ACC mean')
    axis.fill_between(
        target_summary['epoch'],
        target_summary['target_acc_mean'] - target_summary['target_acc_std'],
        target_summary['target_acc_mean'] + target_summary['target_acc_std'],
        alpha=0.2, label='± 1 subject std',
    )
    axis.plot(target_summary['epoch'], target_summary['source_validation_acc_mean'], linestyle='--', label='Source validation ACC mean')
    axis.set(xlabel='Epoch', ylabel='Accuracy', title='SEED-IV RD diagnostic target monitoring')
    axis.grid(alpha=0.3)
    axis.legend()
    figure.tight_layout()
    figure.savefig(RUN_DIR / 'target_accuracy_curve.png', dpi=160)
    plt.show()
except ModuleNotFoundError:
    print('matplotlib is not installed; CSV summaries were still saved.')


## How to interpret a large gap from the old ~90% result

If this notebook remains near 45–55%, compare the old experiment along these axes before changing the model:

1. Did the old code randomly split windows/trials, allowing the same subject or trial into train and test?
2. Was ~90% session-dependent or within-subject rather than subject-independent LOSO?
3. Did the old model use fused DE+RD/gating/graph features rather than RD alone?
4. Was the RD reference fitted globally, including the target subject? That raises accuracy but leaks target information.
5. Were individual one-second windows treated as samples instead of whole trials?

The saved `settings.json`, per-fold CSV files, and target curve make these comparisons auditable.